# Markov Models

In this lab we will use Markov models to study human languages.

## Data Preparation

Before we can actually think about mathematical modeling, we need to gather some sample data to work on. For this lab, we will be retrieving text from the website of [Project Gutenberg](https://www.gutenberg.org/).

Go to that project website and find the page for "The Hound of the Baskervilles" novel. In particular, load the simple text form of the book.

Now define a Python variable `novel` and assign to it the first few paragraphs of Chapter 1.

<div class="alert alert-info">
In Python there is a specific syntax to define multi-line strings. If you use such syntax, you can simply copy and paste the novel text from the Project Gutenberg website.
</div>

In [ ]:
novel = ...

The text that you have just copied it's too complex for our purposes. It contains not only words, but also punctuation marks and empty lines.

At the and of our data preparation we want a string containing only words separated by single spaces. Say we start from:

```
“I have, at least, a well-polished, silver-plated coffee-pot in front of
me,” said he.
```

Our final output should be:

```
i have at least a well polished silver pated coffee pot in front of me said he
```

Your first task is therefore to simplify the data, performing the following steps (not necessarily in this order):
* remove the distinction from upper and lower case, for instance converting everything to lower case;
* remove all characters that are not letters or spaces;
* remove empty lines and redundant spaces (words must be separated by a **single** space).

Remember that Python strings behave like containers. They are collections of single characters that you can iterate upon using `for` loops or list comprehensions.

You may find the following two functions useful for converting strings into actual lists and vice-versa.

In [ ]:
def string2list(s):
    return list(s)

In [ ]:
def list2string(l):
    return ''.join(l)  # join all characters together

And the following two functions for converting strings into list of words and vice-versa.

In [ ]:
def string2words(s):
    return s.split(' ')  # split on spaces

In [ ]:
def words2string(ws):
    return ' '.join(ws)  # join words with a spaces between each pair

Now implement the transformations described above.

In [ ]:
novel_clean = ...

Now that we have cleaned up the text we will be using, there is one more conversion we have to make before we think abount the mathematical modeling.

Each character in our string will be represented as a different _state_ of a Markov model. As states are usually labeled with integer numbers (1, 2 ...), we need a way to convert each letter into an integer.

Start by defining a function which takes the list of all possible characters (which you have used above for the cleanup) and assign to each a different number (starting from 0). For instance, one possible pairing would be:

```
a -> 0
b -> 1
c -> 2
...
```

In [ ]:
def letters2int(ls):
    ...

Now write a second function that:
* converts the cleaned-up text into a list of characters;
* replaces each character with the correct number.

Using the mapping defined above, the string
```
acca
```

would become
```python
[0, 2, 2, 0]
```

In [ ]:
def string2ints(s):
    ...

## Markov Model, order 1

Here we are interested in estimating an order 1 Markov model representing our text.

In practice, we need to build a matrix capturing all the transition probabilities between pairs of characters.

The actual workflow will be we following:
* build a numpy matrix representing transitions between characters of the appropriate size;
* the matrix should be initially filled with zeroes;
* for each pair of characters (say 'a' and 'b') count how many times the pair 'ab' appears in the text and store it in the matrix;
* transform the frequency matrix into a proability matrix.

Implement in Python code the above steps. You may want to start doing one step at a time, interacting with the notebook. Later you should gather all the code into a single function like the following:

```python
def estimate_markov_1(s):  # s is the input text
    # do stuff
    return m  # m is the computed probability matrix
```

In [ ]:
import numpy as np

In [ ]:
def estimate_markov_1(s):  # s in the input text
    ...
    return m  # m is the computed probability matrix

In [ ]:
mm1 = estimate_markov_1(novel_clean)

## Data Probability given a Model

We have now build a model for our text. Unfortunately, it is quite difficult to check whether our computation is correct staring at the model matrix. Since we are nonetheless interested in verifying the correctness of our algorithm, we will adopt a different strategy.

We have estimated our model from a specific piece of text. In statistical terms, we have computed:

$$ P(\theta | D ) $$

where $\theta$ represents the parameters of the model and $D$ the input data.

In this particular case it is quite easy to compute the inverse, that is:

$$ L(D | \theta) $$

the likelihood of a piece of text under the assumption that it was generated by the Markov model.

In computational terms we need to:
* scan the original text;
* for each pair of characters look up the corresponding probability in the model;
* since we assume each pair is an independent event, we can obtain the total probability for the text by multiplying all the intermediate values.

The last step is possible due to the well known probability relation for **independent** events:

$$ P(A,B) = P(A) P(B) $$

There is one last detail we should take care of. The probabilities in our model are going to be quite small. Say $P(A)=0.02$ and $P(B)=0.03$. When we multiply them together, the result is even smaller: $P(A,B) = 0.0006$.

In the long run, this is going to cause numerical problems. The computer is only precise to a certain level; if we get very small values they became essentially indistinguishable from zero.

Luckily, we can use a trick to sidestep this issue: take the logarithm of the probability. By the property of the logarithm, products turn into sums:

$$ \log(\prod_i p_i) = \sum_i \log(p_i) $$

Instead of computing the likelihood directly, we will therefore compute the log-likelihood.

OK, we are now ready to write some code. Implement a function doing the following:
* iterate over all successive pairs in a text;
* look up the corresponding probability in the model;
* take its logarithm;
* return the sum of all logarithms.

<div class="alert alert-warning">
What happens if you have a transition probability of zero? What is its logarithm?
Can you figure a meaningful way to solve this issue?
</div>

In [ ]:
def loglik(ints, m):
    ...

In [ ]:
loglik(string2ints(novel_clean), mm1)

The value of the log-likelihood is again not particularly informative on its own. We need to compute the log-likelihood on a different text to establish a baseline for comparison.

Let's build a random sequence of characters of the same length of the novel fragment we have used. To this end we will use a function of the `numpy` package.

In [ ]:
novel_random = np.random.randint(0, ..., ...)

Now compute the log-likelihood on this new text (you can reuse the function!) and compare it with the previous result.

The Markov model we have generated "explains" better the first (novel) text or the second (random) one?

In [ ]:
loglik(novel_random, mm1)

## Language Detection

In the last section we learned how to measure the likelihood that a given piece of text derives from a given Markov model. We can now put this knowledge to some practical use by implementing a language classificator.

The idea is to build a software that can automatically detect the language (such as Italian or English) of a string provided by the user.

To do that, we first need to train the software with some samples of each language. For English, we already have our novel.

In [ ]:
english_sample = novel_clean

Now go to the Project Gutenberg website and get a sample of French and Italian. Define two more variables, `french_sample` and `italian_sample`, cleaning each text sample in the same way you've done with the english text.

In [ ]:
def clean_sample(s):
    ...

In [ ]:
french_sample = clean_sample(...)

<div class="alert alert-info">
Compare the cleaned-up version of the french text with the original. Do you notice any significant difference?
<br><br>
When you've identified it, contact one of the instructors.
</div>

In [ ]:
def clean_sample(s):
    ...

In [ ]:
french_sample = clean_sample(...)

In [ ]:
italian_sample = clean_sample(...)

Now build three models, one for each sample.

In [ ]:
english_model = estimate_markov_1(english_sample)
french_model = estimate_markov_1(french_sample)
italian_model = estimate_markov_1(italian_sample)

We are ready to build the classifier. Write a function that receives a single string input. It has to compute the log-likelihood of that string according to the different models and then pick the best one. The function return value should be the name of the guessed language.

In [ ]:
def guess_language(s):
    ...

In [ ]:
guess_language("ecco un messaggio scritto in italiano")

In [ ]:
guess_language("a sample message written in plain english gets a good classification")

In [ ]:
guess_language('''
Allons enfants de la Patrie
Le jour de gloire est arrivé!
Contre nous de la tyrannie
''')

## Performance Evaluation

<div class="alert alert-info">
This section is optional.
</div>

No classifier is perfect. Here we should put our function `guess_language` to the test and check in which conditions it works and in which it provides a wrong result.
There are two choices having a primary impact of performances: the size of the training dataset (text) and that of the message to be classified.

Here we will concentrate on the second, that is the message length. Write a function `extract_with_length` to extract a random selection out of a string, given a desired length.

For instance:

```python
extract_with_length('a simple message', 6)
```

should return, each time it is called, a (different) random substring of length 6.

Examples:

```python
>>> extract_with_length('a simple message', 6)
"a simp"
>>> extract_with_length('a simple message', 6)
"messag"
```

<div class="alert alert-info">
Import the ```random``` library. You can then use the ```random.randint``` function to generate random integer numbers. Use the Jupyter documentation to check how the function works.
</div>

In [ ]:
import random

In [ ]:
def extract_with_length(s, n):
    ...

What happens if I run the following code:

```python
extract_with_length('short', 20)
```

How can I avoid this problem, while improving the error message?

We can now implement a Monte Carlo simulation to evaluate the performance of our classifier with respect to the length of the input message.

* choose a length `l` (for instance, `l=10`) and extract 1000 random strings with `extract_with_length`;
* run `guess_language` on those strings and record whether the classifier was right or wrong;
* calculate the performance as the ratio between correct and total classifications;
* repeat the above procedure for all lengths `l` between 5 and 40.

Prepare a scatter plot displaying the value of `l` on the $x$ axis and the performance on the $y$ axis.

Do you see any trend? Can you explain it?

In [ ]:
import matplotlib.pylab as plt
%matplotlib inline

In [ ]:
def performance_monte_carlo(template, expected_language):
    ... 
    plt.plot(X, Y)

In [ ]:
performance_monte_carlo(clean_sample('''
Va’, pensiero, sull'ali dorate;
va’, ti posa sui clivi, sui colli,
ove olezzano tepide e molli
l'aure dolci del suolo natal!

Del Giordano le rive saluta,
di Sionne le torri atterrate...
Oh, mia patria sì bella e perduta!
Oh, membranza sì cara e fatal!

Arpa d'or dei fatidici vati,
perché muta dal salice pendi?
Le memorie nel petto raccendi,
ci favella del tempo che fu!

O simile di Solima ai fati
traggi un suono di crudo lamento,
o t'ispiri il Signore un concento
che ne infonda al patire virtù!
'''), 'italian')

## Appendix A

In [ ]:
import unicodedata
def sa(s):
    return ''.join(c for c in unicodedata.normalize('NFD', s)
                   if unicodedata.category(c) != 'Mn')